# Lab 2 (ต่อ) · Data Profiling ข้อมูลลูกค้า `customers.csv`

ต่อยอดจาก Lab 2.1 (`sales_raw.csv`) มาตรวจข้อมูลสมาชิกของบ้านบรู แล้วเตรียม `customers_clean.csv` ไปแสดงผลบน Dashboard

| ขั้น | ทำอะไร |
|---|---|
| 1 | โหลดและดูข้อมูลครั้งแรก |
| 2 | Profiling: หาความผิดปกติ **แต่ยังไม่แก้** |
| 3 | ตัดสินใจว่าจะจัดการแต่ละเรื่องอย่างไร (รวมเรื่อง PDPA) |
| 4 | ทำความสะอาด + เก็บ log |
| 5–6 | Verify และ export |
| 7 | สรุปตัวเลขที่จะใช้บน Dashboard |

**Prompt ที่ใช้:** “ช่วยทำ data profiling หรือค้นหาความผิดปกติของข้อมูล customers.csv แล้วสร้างเป็นไฟล์สำหรับทำงานบน google colab ดังตัวอย่าง Lab2_1_Data_Cleaning_NK.ipynb”

**ไฟล์ที่ต้องอัปโหลด:** `customers.csv`, `branches.csv` และ `sales_clean.csv` (ผลจาก Lab 2.1 ใช้ตรวจความสอดคล้องกับยอดขาย)

## ขั้นที่ 0 · เตรียมเครื่องมือ
รันเซลล์นี้ก่อน (ไม่ต้องแก้) มีค่ามาตรฐานที่ใช้ตรวจ และ `load_csv()` ที่อ่านทุกคอลัมน์เป็นข้อความ

In [1]:
# ===== เซลล์ตัวช่วย · รันก่อน ไม่ต้องแก้ =====
import os, re
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

GENDERS = ["หญิง", "ชาย", "ไม่ระบุ"]                                  # ค่าเพศที่ยอมรับ
AGE_ORDER = ["ต่ำกว่า 18", "18-24", "25-34", "35-44", "45-54", "55+"]  # ช่วงอายุเรียงจากน้อยไปมาก
DATA_END = "2026-09-20"                                                # วันสุดท้ายของข้อมูลยอดขาย


def load_csv(name):
    """อ่าน CSV ทุกคอลัมน์เป็นข้อความ ถ้าไม่มีไฟล์จะให้อัปโหลด (Google Colab)"""
    if not os.path.exists(name):
        try:
            from google.colab import files
            print(f"ไม่พบ {name} กรุณาอัปโหลดไฟล์")
            files.upload()
        except ImportError:
            raise FileNotFoundError(f"ไม่พบไฟล์ {name} ให้วางไฟล์ไว้โฟลเดอร์เดียวกับ notebook")
    return pd.read_csv(name, dtype=str, keep_default_na=False)

## ขั้นที่ 1 · ดูข้อมูลครั้งแรก

อ่าน **ทุกคอลัมน์เป็นข้อความ** โดยตั้งใจ เหมือน Lab 2.1: ถ้าให้ pandas เดาชนิดข้อมูลเอง รหัสอย่าง `C00001` หรือค่าว่างอาจถูกแปลงจนปัญหาถูกซ่อน

In [2]:
cust = load_csv("customers.csv")
branches = load_csv("branches.csv")
sales = load_csv("sales_clean.csv")   # ผลจาก Lab 2.1 ใช้ตรวจว่าลูกค้าในยอดขายมีอยู่จริงไหม
print(f"customers: {len(cust):,} แถว × {cust.shape[1]} คอลัมน์ · branches: {len(branches)} · sales_clean: {len(sales):,}")
cust.head(8)

customers: 3,000 แถว × 7 คอลัมน์ · branches: 5 · sales_clean: 53,057


,customer_id,nickname,gender,age_group,home_branch_id,joined_date,phone
0,C00001,กมล,หญิง,18-24,B02,2026-03-03,082-xxx-6693
1,C00002,กิตติ,ชาย,25-34,B04,2026-01-18,084-xxx-3734
2,C00003,ปิยะ,หญิง,18-24,B04,2025-05-04,089-xxx-9286
3,C00004,ปิยะ,หญิง,18-24,B05,2026-05-05,083-xxx-7314
4,C00005,รัตนา,หญิง,25-34,B04,2026-07-10,083-xxx-5701
5,C00006,กมล,ชาย,18-24,B04,2026-08-16,086-xxx-1895
6,C00007,วรรณ,ชาย,45-54,B03,2026-07-17,087-xxx-3147
7,C00008,ภัทร,หญิง,25-34,B01,2026-04-07,087-xxx-9877


**คำถามชวนคิด:** ไฟล์นี้มีคอลัมน์ไหนบ้างที่เป็น **ข้อมูลส่วนบุคคล** และถ้าจะเอาขึ้นเว็บ Dashboard จำเป็นต้องใช้คอลัมน์เหล่านั้นจริงไหม

## ขั้นที่ 2 · Profiling: หาความผิดปกติก่อน ยังไม่แก้

**Prompt**
```
ช่วยทำ data profiling ของ DataFrame cust (customers.csv, ทุกคอลัมน์เป็น str ค่าว่างเป็น "")
- จำนวนค่าว่าง (แยก "" กับมีแต่ช่องว่าง) และค่าไม่ซ้ำของแต่ละคอลัมน์
- แถวซ้ำทุกคอลัมน์ และ customer_id ซ้ำ · customer_id ผิดรูปแบบ C + ตัวเลข 5 หลัก
- ค่า gender, age_group, home_branch_id ทั้งหมดพร้อมจำนวน และจำนวนที่ไม่อยู่ในค่าที่ยอมรับ
- joined_date ที่ไม่ใช่ YYYY-MM-DD หรือไม่ใช่วันที่มีจริง, สมัครหลังวันสุดท้ายของข้อมูล, สมัครก่อนสาขาประจำเปิด
- phone ที่ไม่ใช่รูปแบบปิดบัง 0XX-xxx-XXXX (ถ้ามีเบอร์เต็มคือเสี่ยง PDPA) และเบอร์ซ้ำ
- เทียบกับ sales: customer_id ในยอดขายที่ไม่มีในไฟล์ลูกค้า, สมาชิกที่ไม่เคยซื้อ, ซื้อครั้งแรกก่อนวันสมัคร
เก็บจำนวนแต่ละข้อไว้ในตัวแปร n_... อย่าแก้ข้อมูล ให้รายงานอย่างเดียว
```

In [3]:
# ===== ขั้นที่ 2 · Data profiling (รายงานอย่างเดียว ไม่แก้ cust) =====
def is_blank(s):
    """ว่าง = "" หรือมีแต่ช่องว่าง"""
    return s.str.strip().eq("")


def samples(s, n=5):
    """ตัวอย่างค่าไม่ซ้ำ n ค่าแรก ใส่เครื่องหมายคำพูดให้เห็นช่องว่าง"""
    return ", ".join(repr(v) for v in s.drop_duplicates().head(n)) or "-"


def header(title):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")


# 1) ค่าว่างและค่าไม่ซ้ำ
header("1) ค่าว่างและค่าไม่ซ้ำ รายคอลัมน์")
empty = cust.eq("").sum()
profile = pd.DataFrame({
    "empty": empty,
    "blank_space_only": cust.apply(is_blank).sum() - empty,
    "unique": cust.nunique(),
})
print(profile.to_string())
n_blank = int(cust.apply(is_blank).sum().sum())
n_space = int((cust != cust.apply(lambda s: s.str.strip())).sum().sum())
print(f"ค่าว่างทั้งหมด: {n_blank:,} ช่อง · ช่องที่มีช่องว่างหัว/ท้าย: {n_space:,}")

# 2) แถวซ้ำ / customer_id ซ้ำ / รูปแบบรหัส
header("2) แถวซ้ำ และ customer_id")
n_dup_rows = int(cust.duplicated().sum())
n_dup_id = int(cust["customer_id"].duplicated().sum())
n_bad_id = int((~cust["customer_id"].str.match(r"^C\d{5}$")).sum())
print(f"n_dup_rows = {n_dup_rows:,} · n_dup_id = {n_dup_id:,} · n_bad_id = {n_bad_id:,}")

# 3) ค่ากลุ่ม (categorical)
header("3) gender / age_group / home_branch_id")
BRANCH_IDS = list(branches["branch_id"])
for col, allowed in [("gender", GENDERS), ("age_group", AGE_ORDER), ("home_branch_id", BRANCH_IDS)]:
    vc = cust[col].value_counts()
    print(f"{col}: " + " · ".join(f"{'✓' if v in allowed else '✗'} {v!r} {c:,}" for v, c in vc.items()))
n_bad_gender = int((~cust["gender"].isin(GENDERS)).sum())
n_bad_age = int((~cust["age_group"].isin(AGE_ORDER)).sum())
n_bad_branch = int((~cust["home_branch_id"].isin(BRANCH_IDS)).sum())
print(f"n_bad_gender = {n_bad_gender:,} · n_bad_age = {n_bad_age:,} · n_bad_branch = {n_bad_branch:,}")

# 4) joined_date
header("4) joined_date")
jd = pd.to_datetime(cust["joined_date"].where(cust["joined_date"].str.match(r"^\d{4}-\d{2}-\d{2}$")),
                    format="%Y-%m-%d", errors="coerce")
opened = cust["home_branch_id"].map(dict(zip(branches["branch_id"], pd.to_datetime(branches["opened_date"]))))
n_bad_joined = int(jd.isna().sum())
n_joined_future = int((jd > pd.Timestamp(DATA_END)).sum())
n_joined_before_open = int((jd < opened).sum())
print(f"ช่วงวันที่: {jd.min():%Y-%m-%d} ถึง {jd.max():%Y-%m-%d}")
print(f"n_bad_joined = {n_bad_joined:,} · n_joined_future = {n_joined_future:,} · n_joined_before_open = {n_joined_before_open:,}")

# 5) phone (PDPA)
header("5) phone")
masked = cust["phone"].str.match(r"^0\d{2}-xxx-\d{4}$")
full_number = cust["phone"].str.replace(r"\D", "", regex=True).str.len().ge(9)
n_bad_phone = int((~masked).sum())
n_phone_unmasked = int(full_number.sum())
n_dup_phone = int(cust["phone"].duplicated().sum())
print(f"n_bad_phone = {n_bad_phone:,} · n_phone_unmasked (เบอร์เต็ม) = {n_phone_unmasked:,} · n_dup_phone = {n_dup_phone:,}")
print(f"ตัวอย่างเบอร์ซ้ำ: {samples(cust.loc[cust['phone'].duplicated(keep=False), 'phone'])}")
n_minor = int(cust["age_group"].eq("ต่ำกว่า 18").sum())
print(f"สมาชิกอายุต่ำกว่า 18: {n_minor:,} คน")

# 6) เทียบกับยอดขาย
header("6) ความสอดคล้องกับ sales_clean.csv")
member_rows = sales[sales["customer_id"].str.strip() != ""]
sales_ids = set(member_rows["customer_id"])
cust_ids = set(cust["customer_id"])
n_unknown_member = len(sales_ids - cust_ids)
n_no_purchase = len(cust_ids - sales_ids)
first_buy = pd.to_datetime(member_rows.assign(d=member_rows["datetime"].str[:10]).groupby("customer_id")["d"].min())
n_buy_before_join = int((first_buy.reindex(cust["customer_id"]).values < jd.values).sum())
print(f"n_unknown_member (มีในยอดขายแต่ไม่มีในไฟล์ลูกค้า) = {n_unknown_member:,}")
print(f"n_no_purchase (สมาชิกที่ไม่เคยซื้อ) = {n_no_purchase:,} ({n_no_purchase / len(cust):.1%})")
print(f"n_buy_before_join (ซื้อครั้งแรกก่อนวันสมัคร) = {n_buy_before_join:,}")

header("สรุป")
summary = pd.Series({k: globals()[k] for k in [
    "n_blank", "n_space", "n_dup_rows", "n_dup_id", "n_bad_id", "n_bad_gender", "n_bad_age", "n_bad_branch",
    "n_bad_joined", "n_joined_future", "n_joined_before_open", "n_bad_phone", "n_phone_unmasked", "n_dup_phone",
    "n_unknown_member", "n_no_purchase", "n_buy_before_join"]})
print(summary.to_string())


1) ค่าว่างและค่าไม่ซ้ำ รายคอลัมน์
                empty  blank_space_only  unique
customer_id         0                 0    3000
nickname            0                 0      24
gender              0                 0       3
age_group           0                 0       6
home_branch_id      0                 0       5
joined_date         0                 0     530
phone               0                 0    2950
ค่าว่างทั้งหมด: 0 ช่อง · ช่องที่มีช่องว่างหัว/ท้าย: 0

2) แถวซ้ำ และ customer_id
n_dup_rows = 0 · n_dup_id = 0 · n_bad_id = 0

3) gender / age_group / home_branch_id
gender: ✓ 'หญิง' 1,652 · ✓ 'ชาย' 1,232 · ✓ 'ไม่ระบุ' 116
age_group: ✓ '25-34' 993 · ✓ '18-24' 775 · ✓ '35-44' 577 · ✓ '45-54' 346 · ✓ '55+' 196 · ✓ 'ต่ำกว่า 18' 113
home_branch_id: ✓ 'B01' 776 · ✓ 'B05' 693 · ✓ 'B02' 643 · ✓ 'B04' 483 · ✓ 'B03' 405
n_bad_gender = 0 · n_bad_age = 0 · n_bad_branch = 0

4) joined_date
ช่วงวันที่: 2025-04-01 ถึง 2026-09-14
n_bad_joined = 0 · n_joined_future = 0 · n_joined_before_ope

n_unknown_member (มีในยอดขายแต่ไม่มีในไฟล์ลูกค้า) = 0
n_no_purchase (สมาชิกที่ไม่เคยซื้อ) = 493 (16.4%)
n_buy_before_join (ซื้อครั้งแรกก่อนวันสมัคร) = 0

สรุป
n_blank                   0
n_space                   0
n_dup_rows                0
n_dup_id                  0
n_bad_id                  0
n_bad_gender              0
n_bad_age                 0
n_bad_branch              0
n_bad_joined              0
n_joined_future           0
n_joined_before_open      0
n_bad_phone               0
n_phone_unmasked          0
n_dup_phone              50
n_unknown_member          0
n_no_purchase           493
n_buy_before_join         0


### ✔️ อ่านผล
ข้อมูลลูกค้าชุดนี้ **ไม่มีปัญหาเชิงรูปแบบ** (ค่าว่าง รหัสซ้ำ ค่ากลุ่ม วันที่ = 0 ทุกข้อ) ต่างจาก `sales_raw.csv` มาก สิ่งที่ต้องตัดสินใจจริงคือ

* **สมาชิกที่ไม่เคยซื้อ** (`n_no_purchase`) และ **เบอร์โทรซ้ำ** (`n_dup_phone`) ซึ่งเป็นเรื่องความหมายทางธุรกิจ ไม่ใช่ข้อมูลเสีย
* **PDPA:** ไฟล์มี `nickname` และ `phone` ของบุคคล (แม้ปิดบังบางส่วน) และมีสมาชิกอายุต่ำกว่า 18 ปี

> ข้อจำกัด: ไฟล์ที่ได้รับผ่าน Google Drive/Sheets มาแล้ว ถ้าต้นฉบับมีช่องว่างท้ายข้อความ จะถูกตัดไปก่อนถึงมือเรา (เหมือนชื่อสาขาใน Lab 2.1)

## ขั้นที่ 3 · ตัดสินใจก่อนลงมือ

| เรื่องที่พบ | ทางเลือก | ตัดสินใจ | เหตุผล |
|---|---|---|---|
| ค่าว่าง / ช่องว่างหัว-ท้าย | strip / ลบ | strip ทุกคอลัมน์ (ไม่มีผลกับไฟล์นี้) | กันไว้ถ้าได้ไฟล์ต้นฉบับที่มีช่องว่าง |
| แถวซ้ำ / customer_id ซ้ำ | ลบ / เก็บ | ลบแถวซ้ำ ถ้า id ซ้ำแต่ข้อมูลต่างให้รายงาน | 1 คน = 1 แถว |
| ค่ากลุ่มผิด (เพศ/อายุ/สาขา) | map / ลบ | รายงานไว้ ไม่ลบ (ไฟล์นี้ไม่มี) | ไม่เดาค่าแทนลูกค้า |
| joined_date ผิด/อนาคต/ก่อนสาขาเปิด | แก้ / ลบ | รายงานไว้ (ไฟล์นี้ไม่มี) | |
| สมาชิกที่ไม่เคยซื้อ | ลบ / เก็บ | **เก็บ** | ยังเป็นสมาชิกจริง ใช้วัดว่าสมาชิกกี่ % ที่ active |
| เบอร์โทรซ้ำ (ปิดบังแล้ว) | รวมเป็นคนเดียว / เก็บ | **เก็บ ไม่รวม** | เห็นแค่ 4 หลักท้าย เบอร์ต่างคนตรงกันได้ บอกไม่ได้ว่าเป็นคนเดียวกัน |
| `nickname`, `phone` (ข้อมูลส่วนบุคคล) | เก็บ / ตัดออก | **ตัดออกจากไฟล์ที่ขึ้นเว็บ** | หลักใช้ข้อมูลเท่าที่จำเป็น (PDPA) Dashboard ไม่ได้ใช้สองคอลัมน์นี้ |
| สมาชิกอายุต่ำกว่า 18 ปี | เก็บ / ตัด | เก็บเป็นตัวเลขรวมเท่านั้น | แสดงแค่จำนวนต่อช่วงอายุ ไม่แสดงรายบุคคล |

## ขั้นที่ 4 · ทำความสะอาด

**Prompt**
```
เขียนโค้ด pandas ทำความสะอาด cust (อ่านมาเป็น str ทุกคอลัมน์ ค่าว่างเป็น "")
- ทำงานบน clean = cust.copy() ห้ามแก้ cust · เก็บ log (ขั้นตอน, จำนวนแถวที่กระทบ, การตัดสินใจ)
- strip ทุกคอลัมน์ · ลบแถวซ้ำทุกคอลัมน์ · รายงาน customer_id ซ้ำที่ข้อมูลต่างกัน
- รายงาน (ไม่ลบ) ค่า gender/age_group/home_branch_id/joined_date ที่ผิด
- เก็บสมาชิกที่ไม่เคยซื้อ และเบอร์ซ้ำไว้
- ตัดคอลัมน์ nickname และ phone ออก (PDPA: ใช้ข้อมูลเท่าที่จำเป็น)
- ผลลัพธ์คอลัมน์: customer_id, gender, age_group, home_branch_id, joined_date
```

In [4]:
# ===== ขั้นที่ 4 · ทำความสะอาด (ทำบน clean = cust.copy() ไม่แก้ cust) =====
clean = cust.copy()
log = []
rows_before = len(clean)


def add_log(step, n, decision):
    log.append({"ขั้นตอน": step, "จำนวนแถวที่กระทบ": int(n), "การตัดสินใจ": decision})


# 1) ตัดช่องว่างหัว/ท้ายทุกคอลัมน์
stripped = clean.apply(lambda s: s.str.strip())
add_log("1) ตัดช่องว่างหัว/ท้าย", (stripped != clean).any(axis=1).sum(), "strip ทุกคอลัมน์")
clean = stripped

# 2) แถวซ้ำทุกคอลัมน์ และ customer_id ซ้ำ
dup = clean.duplicated()
clean = clean[~dup]
add_log("2) ลบแถวซ้ำทุกคอลัมน์", dup.sum(), "ลบ: 1 คน = 1 แถว")
dup_id = clean["customer_id"].duplicated(keep=False)
add_log("2) customer_id ซ้ำแต่ข้อมูลต่างกัน", dup_id.sum(), "รายงานไว้ ไม่ลบ (ต้องถามทีม CRM ว่าแถวไหนถูก)")

# 3) ค่ากลุ่มและวันที่ที่ผิด — รายงานอย่างเดียว ไม่เดาค่าแทนลูกค้า
bad = {
    "gender": ~clean["gender"].isin(GENDERS),
    "age_group": ~clean["age_group"].isin(AGE_ORDER),
    "home_branch_id": ~clean["home_branch_id"].isin(branches["branch_id"]),
    "joined_date": pd.to_datetime(clean["joined_date"], format="%Y-%m-%d", errors="coerce").isna(),
}
for col, mask in bad.items():
    add_log(f"3) {col} ผิดรูปแบบ/ไม่อยู่ในค่าที่ยอมรับ", mask.sum(), "รายงานไว้ ไม่ลบ (ควรเป็น 0)")

# 4) สมาชิกที่ไม่เคยซื้อ และเบอร์ซ้ำ — เก็บไว้
add_log("4) สมาชิกที่ไม่เคยซื้อ", (~clean["customer_id"].isin(sales_ids)).sum(), "เก็บ: ยังเป็นสมาชิก ใช้วัด % active")
add_log("4) เบอร์โทรซ้ำ (ปิดบังแล้ว)", clean["phone"].duplicated().sum(), "เก็บ ไม่รวมคน: เห็นแค่ 4 หลักท้าย")

# 5) PDPA: ตัดข้อมูลส่วนบุคคลที่ Dashboard ไม่ใช้
PII = ["nickname", "phone"]
add_log("5) ตัดคอลัมน์ nickname, phone", len(clean), "ตัดออก: ใช้ข้อมูลเท่าที่จำเป็น (PDPA)")
clean = clean.drop(columns=PII)

COLS = ["customer_id", "gender", "age_group", "home_branch_id", "joined_date"]
clean = clean[COLS].sort_values("customer_id").reset_index(drop=True)
cleaning_log = pd.DataFrame(log)
print(cleaning_log.to_string(index=False))
print(f"\nจำนวนแถว: ก่อน {rows_before:,} → หลัง {len(clean):,} · คอลัมน์: {', '.join(clean.columns)}")

                                          ขั้นตอน  จำนวนแถวที่กระทบ                                   การตัดสินใจ
                           1) ตัดช่องว่างหัว/ท้าย                 0                              strip ทุกคอลัมน์
                            2) ลบแถวซ้ำทุกคอลัมน์                 0                              ลบ: 1 คน = 1 แถว
               2) customer_id ซ้ำแต่ข้อมูลต่างกัน                 0 รายงานไว้ ไม่ลบ (ต้องถามทีม CRM ว่าแถวไหนถูก)
        3) gender ผิดรูปแบบ/ไม่อยู่ในค่าที่ยอมรับ                 0                   รายงานไว้ ไม่ลบ (ควรเป็น 0)
     3) age_group ผิดรูปแบบ/ไม่อยู่ในค่าที่ยอมรับ                 0                   รายงานไว้ ไม่ลบ (ควรเป็น 0)
3) home_branch_id ผิดรูปแบบ/ไม่อยู่ในค่าที่ยอมรับ                 0                   รายงานไว้ ไม่ลบ (ควรเป็น 0)
   3) joined_date ผิดรูปแบบ/ไม่อยู่ในค่าที่ยอมรับ                 0                   รายงานไว้ ไม่ลบ (ควรเป็น 0)
                           4) สมาชิกที่ไม่เคยซื้อ               493           เก็บ: ยังเ

## ขั้นที่ 5 · Verify
ตรวจว่าไฟล์พร้อมใช้บน Dashboard และไม่มีข้อมูลส่วนบุคคลหลุดไป

In [5]:
def validate_customers(clean):
    ok = True
    def rep(passed, msg):
        nonlocal ok
        ok &= bool(passed)
        print(("✅ " if passed else "❌ ") + msg)

    rep(list(clean.columns) == ["customer_id", "gender", "age_group", "home_branch_id", "joined_date"],
        "คอลัมน์ครบและเรียงถูก")
    rep(not {"nickname", "phone"} & set(clean.columns), "ไม่มีคอลัมน์ข้อมูลส่วนบุคคล (nickname, phone)")
    rep(not clean.apply(lambda s: s.str.contains(r"\d{3}-\w{3}-\d{4}")).any().any(), "ไม่มีรูปแบบเบอร์โทรเหลือในทุกช่อง")
    rep(clean["customer_id"].is_unique, "customer_id ไม่ซ้ำ")
    rep(clean["customer_id"].str.match(r"^C\d{5}$").all(), "customer_id เป็นรูปแบบ C + 5 หลักทุกแถว")
    rep(clean["gender"].isin(GENDERS).all(), f"gender อยู่ใน {GENDERS}")
    rep(clean["age_group"].isin(AGE_ORDER).all(), "age_group อยู่ใน 6 ช่วงอายุ")
    rep(clean["home_branch_id"].isin(branches["branch_id"]).all(), "home_branch_id มีอยู่จริงใน branches.csv")
    jd = pd.to_datetime(clean["joined_date"], format="%Y-%m-%d", errors="coerce")
    rep(jd.notna().all() and (jd <= pd.Timestamp(DATA_END)).all(), f"joined_date เป็นวันที่จริง ไม่เกิน {DATA_END}")
    rep(set(sales_ids) <= set(clean["customer_id"]), "สมาชิกทุกคนในยอดขายมีอยู่ในไฟล์ลูกค้า")
    rep(len(clean) == len(cust) - int(cust.duplicated().sum()), "ไม่มีสมาชิกหายนอกจากแถวซ้ำ")
    print("\n🎉 พร้อม export" if ok else "\nยังมีจุดต้องแก้")
    return ok


validate_customers(clean)

✅ คอลัมน์ครบและเรียงถูก
✅ ไม่มีคอลัมน์ข้อมูลส่วนบุคคล (nickname, phone)
✅ ไม่มีรูปแบบเบอร์โทรเหลือในทุกช่อง
✅ customer_id ไม่ซ้ำ
✅ customer_id เป็นรูปแบบ C + 5 หลักทุกแถว
✅ gender อยู่ใน ['หญิง', 'ชาย', 'ไม่ระบุ']
✅ age_group อยู่ใน 6 ช่วงอายุ
✅ home_branch_id มีอยู่จริงใน branches.csv
✅ joined_date เป็นวันที่จริง ไม่เกิน 2026-09-20
✅ สมาชิกทุกคนในยอดขายมีอยู่ในไฟล์ลูกค้า
✅ ไม่มีสมาชิกหายนอกจากแถวซ้ำ

🎉 พร้อม export


True

## ขั้นที่ 6 · Export
`customers_clean.csv` นำไปวางใน `public/customers.csv` ของ Dashboard ส่วน `customers_cleaning_log.csv` คือหลักฐานว่าทำอะไรกับข้อมูลไปบ้าง

In [6]:
clean.to_csv("customers_clean.csv", index=False, encoding="utf-8-sig")
cleaning_log.to_csv("customers_cleaning_log.csv", index=False, encoding="utf-8-sig")
print("บันทึก customers_clean.csv และ customers_cleaning_log.csv แล้ว")

try:
    from google.colab import files
    files.download("customers_clean.csv")
    files.download("customers_cleaning_log.csv")
except ImportError:
    pass

บันทึก customers_clean.csv และ customers_cleaning_log.csv แล้ว


## ขั้นที่ 7 · สรุปตัวเลขที่จะแสดงบน Dashboard
ใช้ตรวจว่าตัวเลขบนเว็บตรงกับที่คำนวณด้วย pandas (Verify ห้ามข้าม)

In [7]:
sales["revenue"] = pd.to_numeric(sales["qty"]) * pd.to_numeric(sales["unit_price"])
is_member = sales["customer_id"].str.strip() != ""
mem, walk = sales[is_member], sales[~is_member]
print(f"สมาชิกทั้งหมด {len(clean):,} · ซื้อแล้ว {len(sales_ids):,} ({len(sales_ids) / len(clean):.1%})")
print(f"ยอดขายจากสมาชิก {mem['revenue'].sum() / sales['revenue'].sum():.1%} ของยอดขายทั้งหมด")
print(f"ยอดเฉลี่ยต่อบิล: สมาชิก ฿{mem['revenue'].sum() / mem['order_id'].nunique():,.2f} · "
      f"ลูกค้าทั่วไป ฿{walk['revenue'].sum() / walk['order_id'].nunique():,.2f}")

print("\nสมาชิกตามช่วงอายุ:")
print(clean["age_group"].value_counts().reindex(AGE_ORDER).to_string())
print("\nสมาชิกตามเพศ:", clean["gender"].value_counts().to_dict())
print("\nสมาชิกใหม่รายเดือน:")
print(clean["joined_date"].str[:7].value_counts().sort_index().to_string())

name = dict(zip(branches["branch_id"], branches["branch"]))
bills = mem.drop_duplicates("order_id").merge(clean[["customer_id", "home_branch_id"]], on="customer_id")
bills["at_home"] = bills["branch"] == bills["home_branch_id"].map(name)
home = clean["home_branch_id"].value_counts().rename("members").to_frame()
home["ซื้อที่สาขาประจำ"] = bills.groupby("home_branch_id")["at_home"].mean().map("{:.1%}".format)
home.index = home.index.map(name)
print("\nสมาชิกตามสาขาประจำ:")
print(home.to_string())

สมาชิกทั้งหมด 3,000 · ซื้อแล้ว 2,507 (83.6%)
ยอดขายจากสมาชิก 46.5% ของยอดขายทั้งหมด
ยอดเฉลี่ยต่อบิล: สมาชิก ฿129.41 · ลูกค้าทั่วไป ฿127.46

สมาชิกตามช่วงอายุ:
age_group
ต่ำกว่า 18    113
18-24         775
25-34         993
35-44         577
45-54         346
55+           196

สมาชิกตามเพศ: {'หญิง': 1652, 'ชาย': 1232, 'ไม่ระบุ': 116}

สมาชิกใหม่รายเดือน:
joined_date
2025-04    138
2025-05    140
2025-06    142
2025-07    142
2025-08    144
2025-09    148
2025-10    145
2025-11    175
2025-12    197
2026-01    190
2026-02    161
2026-03    189
2026-04    201
2026-05    200
2026-06    182
2026-07    210
2026-08    215
2026-09     81

สมาชิกตามสาขาประจำ:
                members ซื้อที่สาขาประจำ
home_branch_id                          
สยาม                776            89.1%
มหาวิทยาลัย         693            83.7%
สีลม                643            88.5%
บางนา               483            90.0%
อารีย์              405            91.4%
